# **SYNAPSE - Naina Core Inference Engine on Kaggle (with GPU support)**
This notebook demonstrates how to set up, bypass hardware requirements (microphone, speakers, camera, VLC), and run the **Synapse** AI Engine on a Kaggle GPU instance.

### **System Requirements**
* This notebook works best on **Kaggle GPU T4 x2** or **GPU P100** environments.
* Make sure **Internet** access is turned ON in the Kaggle settings panel on the right.

## **1. Install System and Python Dependencies**
We need to install some system tools (like `ffmpeg`, `espeak-ng` for audio handling, and `portaudio19-dev` to compile PyAudio) and the required Python packages.

In [ ]:
# Install system dependencies (including portaudio19-dev for compiling PyAudio)
!apt-get update && apt-get install -y ffmpeg espeak-ng libasound2-dev libportaudio2 portaudio19-dev

# Install python requirements (excluding heavy pre-installed stuff like torch/scipy to save time)
!pip install pyaudio faster-whisper kokoro-onnx soundfile openwakeword yt-dlp ytmusicapi thefuzz colorama uvicorn fastapi websockets python-dotenv

## **2. Set up Ollama (GPU-Accelerated Local LLM)**
We will download the official Ollama Linux binary, serve the daemon in the background, and pull the **Qwen 2.5 3B Instruct** model used by the assistant.

In [ ]:
import subprocess
import time
import os

# Download and unpack Ollama
print("Installing Ollama...")
!curl -L https://ollama.com/download/ollama-linux-amd64.tar.gz -o ollama.tar.gz
!tar -xzf ollama.tar.gz -C /usr

# Start Ollama service in the background
print("Starting Ollama background process...")
with open("ollama_server.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file, preexec_fn=os.setsid)

# Wait for the service to start
time.sleep(5)

# Pull the Qwen 2.5 3B instruct model
print("Downloading qwen2.5:3b-instruct (takes ~1.9 GB)...")
!ollama pull qwen2.5:3b-instruct
print("Ollama service is up and running!")

## **3. Verify GPU Acceleration**
Verify that PyTorch and Ollama can access the NVIDIA GPUs.

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    
# Check Ollama GPU status
!ollama list

## **4. Setup Synapse Project Path**
We will add the Python path and import the Synapse module.

In [ ]:
import sys
import os

# Make sure we add python to the path
# If you cloned Synapse into '/kaggle/working/Synapse', set that as working directory
project_dir = os.path.abspath(os.getcwd())
if os.path.exists(os.path.join(project_dir, 'python')):
    sys.path.append(project_dir)
    print(f"Added project root {project_dir} to sys.path")
else:
    print("⚠️ Python folder not found in current directory! Please navigate to the Synapse directory first.")

## **5. Initialize the Synapse Engine**
Initialize the AI engine. Since it's the first boot, it will automatically download the Whisper transcription models (~1.5 GB) and Kokoro speech synthesis models (~100 MB) into the local cache.

In [ ]:
import os

# Clean up environment variables if any
os.environ.pop("SYNAPSE_AUDIO_INPUT", None)

from python.engine.main_kaggle import Synapse

# Initialize Synapse (Headless components will automatically bypass microphone and speakers)
app = Synapse()

## **6. Interactive Testing via Text Input Fallback**
Since Kaggle doesn't have a physical microphone, the STT engine automatically falls back to Jupyter's interactive text input box! Run the cell below to chat with Naina.

In [ ]:
# Run a single turn of conversation
print("🤖 Chatting with Naina (type 'exit' to stop)...")
try:
    # listen() falls back to keyboard input if no mic is found
    audio, query = app.ear.listen()
    
    if query:
        print(f"User Query: {query}")
        
        # Run LLM brain
        response = app.brain.run_agentic_llm(query)
        print(f"Naina Response: {response}")
        
        # Speak (saves to naina_response.wav in headless environments)
        app.mouth.speak(response)
except Exception as e:
    print("Error during chat:", e)

## **7. Hardware Bypass Demo: Audio File Input (`input.wav`)**
Let's test the microphone bypass. You can run the code below to synthesize a query to `query.wav` or feed any audio file. We'll set the `SYNAPSE_AUDIO_INPUT` environment variable to point to it, which forces the STT engine to read and transcribe this file instead of the microphone.

In [ ]:
# 1. Synthesize a dummy audio file using Kokoro or save one to test the bypass
# Let's generate a wav file of user asking "What is the weather like in New York?"
# by using Kokoro to write a WAV file so we can feed it into Whisper!
print("Generating a test query audio file...")
samples, sample_rate = app.mouth._kokoro.create("What is the weather like in New York?", voice="af_sarah")
import soundfile as sf
sf.write("query.wav", samples, sample_rate)
print("Generated query.wav containing voice command: 'What is the weather like in New York?'")

# 2. Tell Synapse to read this file instead of the microphone
os.environ["SYNAPSE_AUDIO_INPUT"] = "query.wav"
os.environ["SYNAPSE_AUDIO_OUTPUT_PATH"] = "naina_response.wav"

print("\n👂 Feeding query.wav into the Synapse STT engine...")
audio, query = app.ear.listen()
print(f"✓ Transcribed Query: '{query}'")

if query:
    # 3. Process query through the Agent LLM
    response = app.brain.run_agentic_llm(query)
    print(f"✓ Naina Brain Output: '{response}'")
    
    # 4. Synthesize voice response
    app.mouth.speak(response)

## **8. Play Naina's Response Audio**
Since Kaggle is headless, the TTS Engine saved Naina's spoken voice response to `naina_response.wav`. You can play the synthesized speech directly inside the notebook below!

In [ ]:
from IPython.display import Audio, display

if os.path.exists("naina_response.wav"):
    print("🔊 Playing Naina's voice response:")
    display(Audio("naina_response.wav", autoplay=True))
else:
    print("⚠️ naina_response.wav not found. Make sure you ran Section 7 successfully!")